# 第二章：机器学习基础

## 在学习神经网络之前，必须弄清楚的 27 个核心概念

本章不教具体的模型，只给后续所有章节奠定共同语言。每个概念的定义、直觉、公式和例子在同一页内完成——以后遇到不再重复解释，直接引用。

分三组：数学前提 → ML 术语 → 补充概念。

## 2.1 数学前提

以下概念是所有公式的基础。你不会的每一个，都会在后面某章变成拦路石。

### 2.1.1 向量、矩阵与张量

**定义：**

- **标量 (Scalar)**：一个单独的数。$s \in \mathbb{R}$（实数）。例：$3.14$，$-2$。
- **向量 (Vector)**：一列有序的数。$\mathbf{v} \in \mathbb{R}^n$ 表示 $n$ 维向量。例：$\mathbf{v} = [1, 2, 3]^T$ 是 3 维列向量。
- **矩阵 (Matrix)**：一个二维数表。$\mathbf{M} \in \mathbb{R}^{m \times n}$ 表示 $m$ 行 $n$ 列的矩阵。
- **张量 (Tensor)**：三维及以上的多维数组。$\mathcal{T} \in \mathbb{R}^{a \times b \times c}$。

**直觉：** 向量 = 一个数据点的所有特征值排成一列。矩阵 = 多个数据点叠在一起（每行一个样本）。张量 = 矩阵的矩阵（如图片批：batch×高×宽×通道）。

**运算：**

- **转置** $\mathbf{M}^T$：行列互换。
- **点积** $\mathbf{a} \cdot \mathbf{b} = \sum_i a_i b_i$：两个向量的对应元素相乘再求和，结果是一个标量。度量两个向量的"方向一致程度"——同向时点积大，正交时点积为零，反向时点积为负。
- **矩阵乘法** $(\mathbf{A}\mathbf{B})_{ij} = \sum_k a_{ik} b_{kj}$：$\mathbf{A}$ 的第 $i$ 行与 $\mathbf{B}$ 的第 $j$ 列做点积。

**在 ML 中的核心用途：** 一个全连接层 $\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}$ 就是矩阵乘法加向量加法。一个 batch 的 128 个样本 $\mathbf{X}^{128 \times 784}$ 通过权重矩阵 $\mathbf{W}^{784 \times 256}$ 变成 $\mathbf{X}\mathbf{W}^{128 \times 256}$——一次矩阵乘法完成 128 个样本的前向计算。

### 2.1.2 梯度与偏导数

**定义：**

- **偏导数** $\frac{\partial f}{\partial x_i}$：多元函数 $f(x_1, \ldots, x_n)$ 对单个变量 $x_i$ 的导数——其他变量视为常数时，$x_i$ 变化一个单位，$f$ 变化多少。
- **梯度** $\nabla_{\mathbf{x}} f = \left[\frac{\partial f}{\partial x_1}, \ldots, \frac{\partial f}{\partial x_n}\right]^T$：所有偏导数组成的向量。

**直觉：** 梯度指向函数值上升最快的方向。站在山上，梯度指向最陡的上坡方向；梯度的反方向指向最陡的下坡方向——这就是梯度下降。

**一维类比：** $f(x) = x^2$ 在 $x=2$ 处的导数是 $4$——每向右挪一步，$f$ 增大约 $4$ 倍步长。如果是 $x=-2$，导数是 $-4$，向右挪一步 $f$ 反而减小。梯度只是把这种"单个方向的变化率"推广到多维。

**链式法则：** $\frac{\partial f}{\partial x} = \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial x}$。复合函数的导数 = 外层导数 × 内层导数。神经网络是 $f(g(h(\ldots(\mathbf{x}))))$ 的嵌套——梯度像多米诺骨牌一样从输出层逐层传回输入层，每一步乘以本层的局部导数。这就是反向传播的全部。

### 2.1.3 概率分布、期望与方差

**定义：**

- **随机变量** $X$：取值不确定但遵循某种规律的量。
- **概率分布**：描述随机变量取各值的概率。离散型用 PMF，连续型用 PDF。
- **期望** $\mathbb{E}[X] = \sum_i x_i P(X=x_i)$：随机变量的"加权平均"。
- **方差** $\text{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2]$：数据分散程度的度量。
- **协方差** $\text{Cov}(X,Y) = \mathbb{E}[(X-\mathbb{E}[X])(Y-\mathbb{E}[Y])]$：两个变量"同涨同跌"的程度。

**ML 中的三大核心分布：**

| 分布 | 记号 | 适用场景 |
|------|------|----------|
| 高斯 (正态) | $\mathcal{N}(\mu, \sigma^2)$ | 连续值的噪声模型 |
| 伯努利 | $\text{Bernoulli}(p)$ | 抛硬币式的二值事件 |
| 类别 | $\text{Cat}(p_1, \ldots, p_K)$ | 多分类（K 个互斥类别） |

**为什么方差分母是 $n-1$？** 用样本均值 $\bar{x}$ 替代真实均值 $\mu$ 会低估方差——$\bar{x}$ 被数据"拉"到中心，用它算出的偏差偏小。数学推导给出 $\mathbb{E}[s_n^2] = \frac{n-1}{n}\sigma^2 < \sigma^2$。除以 $n-1$ 校正这个系统性低估（无偏估计）。$n$ 大时 $n$ 和 $n-1$ 差异可忽略。

### 2.1.4 最大似然估计 (MLE)

**问题：** 有一堆观测数据，想推断背后是哪个概率分布产生的。

**定义：** 给定数据 $\mathcal{D} = \{x_1, \ldots, x_n\}$ 和参数化的分布族 $p(x|\theta)$，MLE 选择使观测数据"最可能发生"的参数：

$$\hat{\theta}_{\text{MLE}} = \arg\max_{\theta} \prod_{i=1}^{n} p(x_i|\theta) = \arg\max_{\theta} \sum_{i=1}^{n} \log p(x_i|\theta)$$

**直觉：** 抛 10 次硬币，7 正 3 反。$p=0.7$ 时看到这组数据的概率是 $0.267$，$p=0.5$ 时只有 $0.117$——MLE 选 $p=0.7$。求导可知最优解就是频率 $7/10$。

**MLE 与损失函数的关系：** MSE 等价于假设高斯噪声下的 MLE；交叉熵等价于假设类别分布下的 MLE。损失函数不是凭空定义的——每个常见的损失函数背后都有一个 MLE 解释。

### 2.1.5 熵、KL 散度与交叉熵

**熵** $H(P) = -\sum_i P(i) \log P(i)$：用分布 $P$ 编码事件平均需要多少比特。$P$ 越均匀，$H$ 越大。

**KL 散度** $D_{KL}(P\|Q) = \sum_i P(i) \log \frac{P(i)}{Q(i)}$：用 $Q$ 来近似 $P$ 时多用了多少比特。$D_{KL} \ge 0$，且 $D_{KL}=0$ 当且仅当 $P=Q$。**不对称：** $D_{KL}(P\|Q) \neq D_{KL}(Q\|P)$——$P$ 大概率的事件在 $Q$ 中被严重低估时惩罚极重。相反方向：$P$ 小概率的事件 $Q$ 认为会发生，惩罚轻。

**交叉熵** $H(P, Q) = -\sum_i P(i) \log Q(i) = H(P) + D_{KL}(P\|Q)$：$H(P)$ 是常数，所以**最小化交叉熵 = 最小化 KL 散度**。

**数值例子：** 真实标签 $P = [0.9, 0.1]$，模型预测 $Q = [0.5, 0.5]$（模糊），$D_{KL} \approx 0.37$。如果模型猜反了 $Q = [0.1, 0.9]$：$D_{KL} \approx 1.76$——猜反比猜模糊严重得多。

**ML 中 $\log$ 的底：** 始终是 $e$（自然对数）。因为 $\ln(x)$ 的导数是 $1/x$，用其他底会多一个常数因子。写 $\log$ 不写 $\ln$ 是领域习惯。

### 2.1.6 拉格朗日乘子法

**问题：** 在"必须满足某个条件"的前提下求最优解。

**构造规则：** 对于"最大化 $f(\mathbf{x})$，满足 $g(\mathbf{x}) = 0$"：

$$\mathcal{L}(\mathbf{x}, \lambda) = f(\mathbf{x}) - \lambda \cdot g(\mathbf{x})$$

其中 $\lambda$ 是拉格朗日乘子——它的取值会在优化过程中自动迫使约束成立。

**直觉：** 最优解处，目标函数 $f$ 的梯度方向必须与约束函数 $g$ 的梯度方向平行——因为如果 $f$ 的梯度在约束曲面的切方向上有分量，沿着那个方向走一步就可以在满足约束的同时提高 $f$ 值。$\nabla f = \lambda \nabla g$ 就是这个条件的数学表达。

**ML 中的应用：** PCA 推导中，最大化投影方差 $\mathbf{w}^T\mathbf{C}\mathbf{w}$ 且约束 $\mathbf{w}^T\mathbf{w}=1$，拉格朗日函数推导出特征方程 $\mathbf{C}\mathbf{w}=\lambda\mathbf{w}$。

### 2.1.7 特征值与特征向量

**定义：** 对于方阵 $\mathbf{A}$，如果存在非零向量 $\mathbf{v}$ 和标量 $\lambda$ 满足 $\mathbf{A}\mathbf{v} = \lambda \mathbf{v}$，则 $\lambda$ 是特征值，$\mathbf{v}$ 是特征向量。

**直觉：** 矩阵乘法通常会把向量旋转到另一个方向。但对于特征向量，矩阵只做了一件事——沿着它本身的方向拉伸/压缩 $\lambda$ 倍。特征向量是矩阵变换下的"不动方向"。

**ML 中的核心应用：** PCA 的主成分就是协方差矩阵的特征向量。协方差矩阵描述数据在各方向上的分散程度——它的特征向量恰好就是方差最大的方向，对应的特征值等于沿该方向的方差大小。

**补充：** 非方阵没有特征值——特征值只对方阵定义。PCA 中协方差矩阵恰好是方阵（$d \times d$ 的对称矩阵），所以适用。

### 2.1.8 L1 与 L2 范数

**定义：**
- **L1 范数** $\|\mathbf{w}\|_1 = \sum_i |w_i|$：所有权重的绝对值之和。
- **L2 范数** $\|\mathbf{w}\|_2 = \sqrt{\sum_i w_i^2}$：欧几里得距离。

**直觉：** L2 惩罚大权重（平方放大偏差），迫使用权值均匀分布。L1 的绝对值函数在零点处不光滑，"鼓励"某些权重精确归零——产生稀疏解。

**几何视角：** L2 正则项的等高线是圆形，L1 是菱形。菱形有尖角在坐标轴上，目标函数的等高线最先碰到这些尖角——尖角处某个权重恰好为零。这就是 L1 产生稀疏性的几何原因。

**在 ML 中：** L2 正则化 = 权重衰减——最常见的防止过拟合手段。L1 正则化 = 特征选择——自动挑出重要的特征，丢掉不重要的。Elastic Net = L1 + L2 混合。

## 2.2 ML 基础术语

以下概念定义了这个领域的"词汇表"。后面的每一章都会频繁用到，这里统一讲清楚。

### 2.2.1 监督学习、无监督学习与强化学习

**监督学习**：数据有标签。给定输入 $\mathbf{x}$，预测输出 $y$。
- **分类**：$y$ 是离散类别。
- **回归**：$y$ 是连续数值。

**无监督学习**：数据无标签。发现数据本身的结构：
- **聚类**：把相似的样本分到一组（K-Means, DBSCAN）。
- **降维**：压缩维度但保留主要信息（PCA, t-SNE）。
- **密度估计**：学出数据本身的概率分布（VAE）。

**强化学习**：智能体与环境交互获得奖励信号来学习策略。

**自监督学习**：从数据本身构造"伪标签"——掩掉部分内容让模型预测被掩掉的部分。BERT 和 GPT 的预训练都基于自监督。

### 2.2.2 样本、特征、标签、维度

**样本**：数据集中的一条记录。一张图片、一行表格数据、一句话。

**特征**：描述样本的一个属性。一张 28×28 灰度图有 784 个特征（每个像素的亮度）。

**标签**：我们希望预测的目标值。分类问题是类别号，回归问题是连续值。

**维度**：特征的个数。一个样本有 $d$ 个独立的属性，就是 $d$ 维。文本 Embedding 的 256 维向量中，每个维度没有人类可理解的名称，但编码了语义信息。

$$\mathbf{x} \in \mathbb{R}^d \quad \text{（d 维特征向量）} \qquad y \in \{\text{标签空间}\} \quad \text{（目标）}$$

**注：** "维度灾难"指维度越高，数据点之间的距离趋于相等，需要指数级增长的数据量来维持模型性能。

### 2.2.3 训练集、验证集、测试集

| 集合 | 用途 | 反馈给模型？ |
|------|------|------------|
| 训练集 (Train) | 更新模型参数 | ✓ 直接用于梯度更新 |
| 验证集 (Validation) | 调超参数、选模型 | ✓ 间接（选模型时参考） |
| 测试集 (Test) | 评估最终性能 | ✗ 绝对不可接触 |

**为什么必须有三分？** 训练误差是乐观估计——模型可能死记硬背了训练集。验证集是"没见过但可以频繁看"的数据——用来客观判断模型是否在进步。测试集只在最终评测用一次——一旦你用测试集调过参数，测试集就失去了"公正裁判"的意义。

### 2.2.4 Epoch、Batch 与 Iteration

- **Epoch**：整个训练集被模型完整看过一遍。60,000 张 MNIST 图片 = 1 epoch。
- **Batch**：每次喂给模型的一小批样本。batch_size = 128 表示每次处理 128 张图。
- **Iteration (Step)**：一次参数更新。1 iteration = 处理 1 个 batch 的前向+反向+参数更新。

$$\text{1 epoch 的 iteration 数} = \left\lceil \frac{\text{训练集大小}}{\text{batch_size}} \right\rceil$$

**为什么用 Mini-batch？** 全量→梯度精确但内存爆炸。单样本→快但不稳定，梯度抖动大。Mini-batch (32~256)→平衡快与稳定。太小→梯度噪声大但泛化好；太大→收敛到尖锐局部最优。甜蜜点通常是 32~128。

### 2.2.5 参数 vs 超参数

**参数**：模型从数据中学到的量。权重 $\mathbf{W}$ 和偏置 $\mathbf{b}$——梯度下降自动调。

**超参数**：训练前设定的量。学习率 $\eta$、batch_size、网络层数、每层神经元数、正则化系数 $\lambda$、epoch 数。

**最重要的超参数——学习率**：每次梯度下降的步长 $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L$。太大→训练震荡甚至发散；太小→训练缓慢甚至卡住。实践中用自适应优化器（Adam）让学习率自动调节。

### 2.2.6 损失函数与优化器

**损失函数** $\mathcal{L}(\hat{y}, y)$：度量预测与真实标签的差异。

| 任务 | 常用损失 | 公式 |
|------|---------|------|
| 回归 | MSE | $\mathcal{L} = (\hat{y} - y)^2$ |
| 二分类 | BCE | $\mathcal{L} = -[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ |
| 多分类 | CE | $\mathcal{L} = -\sum_k y_k \log \hat{y}_k$ |

MSE 等价于高斯噪声下的 MLE；CE 等价于类别分布下的 MLE。

**优化器**：根据梯度更新参数的算法。SGD 最简单，Adam 是当前默认选择（自适应调节每个参数的学习率），AdamW 是 Adam + 解耦的权重衰减。

### 2.2.7 过拟合与欠拟合

**欠拟合**：模型太简单，训练误差和验证误差都高。加层、加参数。

**过拟合**：模型太复杂，训练误差低但验证误差高——模型"死记硬背"而非"理解规律"。

**诊断**：画训练误差和验证误差曲线。两条曲线分叉时开始过拟合。

**解决手段（由轻到重）：** (1) 加数据，(2) 正则化（L2/L1/Dropout），(3) Early Stopping，(4) 减少模型容量。

**正则化的统一视角：** 所有正则化方法的本质都是**限制模型的自由度**——不让参数自由生长到任意大的值。

### 2.2.8 归一化与标准化

- **归一化**：把数据缩放到 $[0, 1]$ 或 $[-1, 1]$。$x' = \frac{x - \min}{\max - \min}$
- **标准化**：把数据变成均值为 0、标准差为 1。$x' = \frac{x - \mu}{\sigma}$

**为什么要做：** 神经网络对输入尺度敏感。年龄（0-100）和年收入（0-1,000,000）这两个特征——收入对应的梯度比年龄大一万倍，优化器会忽略年龄维度。标准化后两者在同一数量级上，梯度更新公平。

**BatchNorm**：在网络中间层做的标准化。**LayerNorm**：沿特征维度归一化（Transformer 中用）。

### 2.2.9 分类评估指标

**为什么准确率不够？** 99 个正类、1 个负类——模型永远预测正类，准确率 99%，但对负类完全无能为力。

**混淆矩阵：**

| | 预测为正 | 预测为负 |
|---|---------|---------|
| 实际为正 | TP | FN |
| 实际为负 | FP | TN |

$$\text{Accuracy} = \frac{TP+TN}{\text{all}} \quad \text{Precision} = \frac{TP}{TP+FP} \quad \text{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**选择指南：** 垃圾邮件检测→要高 Precision（别把重要邮件错判为垃圾）。癌症筛查→要高 Recall（别漏掉真的患者）。

### 2.2.10 Softmax 与 Sigmoid

**Sigmoid** $\sigma(x) = \frac{1}{1+e^{-x}}$：把任意实数压到 $(0, 1)$——用于二分类的最后一层。

**Softmax** $\text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$：把一个向量变成概率分布（和为 1）——用于多分类的最后一层。

**两者的关系：** 二分类时，Softmax 退化为 Sigmoid：$\text{softmax}([z, 0])_1 = \frac{e^z}{e^z + 1} = \sigma(z)$。

**梯度饱和：** Sigmoid 的导数 $\sigma'(x) = \sigma(x)(1-\sigma(x))$——当 $x$ 很大或很小时，$\sigma'(x) \to 0$，梯度消失。ReLU 取代 Sigmoid 正是因为 ReLU 在正半轴梯度恒为 1，不饱和。

## 2.3 补充概念

以下概念已在笔记后面章节中散落出现，这里做统一的、精确的定义。

### 2.3.1 前向传播与反向传播

**前向传播**：输入逐层流经网络，每层做 $\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$，最终输出预测。

**反向传播**：计算损失对每个参数的梯度，从输出向输入逐层应用链式法则。

**为什么必须反向算：** 链式法则中，第 $l$ 层的梯度依赖于第 $l+1$ 层的梯度。从输出开始，每往前推一层，累积乘一个局部雅可比矩阵。

**梯度清零 (zero_grad)：** PyTorch 中每次 `backward()` 会**累加**梯度（而非覆盖）。不清零→第二轮更新会用第一轮+第二轮的梯度之和。每次迭代前必须 `optimizer.zero_grad()`。

### 2.3.2 生成模型 vs 判别模型

**判别模型**：直接学 $P(y|\mathbf{x})$——给定输入，输出标签概率。只管"决策边界"。

**生成模型**：学 $P(\mathbf{x}, y)$ 或 $P(\mathbf{x})$——数据本身的分布。学到后可以生成新的、逼真的样本。VAE、GAN、扩散模型都是生成模型。

**GAN 中的生成器与判别器：** 生成器 $G(\mathbf{z})$ 从噪声生成假样本——隐式地建模数据分布。判别器 $D(\mathbf{x})$ 判断输入是真还是假——它是二分类器。两者对抗训练：生成器学生成更逼真的假图，判别器学分辨真伪。

### 2.3.3 注意力机制 (Attention)

**问题：** RNN 必须把整个句子压缩成一个固定长度的隐藏向量——长距离依赖信息容易丢失。

**Self-Attention：** 每个词产生三个向量——Query (我在找什么？)、Key (我是什么？)、Value (我的内容是什么？)。

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

相似度 = $Q \cdot K^T$，用 softmax 变成注意力权重，加权求和所有 Value。除以 $\sqrt{d_k}$ 防止点积值过大导致 softmax 梯度饱和。详细推导见 8.1 节。

### 2.3.4 Embedding：离散符号 → 向量

**问题：** "猫"和"狗"语义相关，但 one-hot 编码时两者距离完全相同——编码本身不携带语义信息。

**Embedding：** 一个可学习的查找表 $\mathbf{E} \in \mathbb{R}^{V \times d_e}$——第 $i$ 行是第 $i$ 个词的 $d_e$ 维向量。

**训练方法 (Word2Vec Skip-gram)：** 给定中心词，预测周围上下文词。这个任务迫使语义相近的词映射到相近的向量——因为相近的向量会产生相似的上下文预测结果。

**在 LLM 中：** GPT 的 Embedding 层将每个 token ID 映射为稠密向量，作为 Transformer 的输入。

### 2.3.5 聚类：无标签的分组

**定义：** 把没有标签的数据点按彼此相似度自动分成若干组。与分类的区别：分类有标签，聚类没有标签——只发现数据本身的结构。

**K-Means 算法：** (1) 随机选 K 个中心点；(2) 把每个数据点分配给最近的中心；(3) 重新计算每个簇的中心（均值）；(4) 重复直到中心不再移动。

**与降维的关系：** 先用 PCA/t-SNE 降到 2 维可视化，再用 K-Means 在高维空间中发现自然分组。降维是"看"，聚类是"分"。

**选择 K：** 肘部法则——画出簇内距离之和 (inertia) 随 K 变化的曲线。曲线拐弯处就是最合适的 K。

### 2.3.6 感受野 (Receptive Field)

**定义：** CNN 中第 $L$ 层的一个神经元在原始输入图像上能"看到"的区域大小。

$$\text{RF}_l = \text{RF}_{l-1} + (k_l - 1) \times \prod_{i=1}^{l-1} s_i$$

**数值例子：** 输入 32×32 → 第 1 层 3×3 卷积 (RF=3×3) → 2×2 池化 (RF=6×6) → 第 2 层 3×3 卷积 (RF=10×10) → 2×2 池化 (RF=22×22) → 第 3 层 3×3 卷积 (RF=26×26)。

**工程意义：** 浅层神经元只能看到局部纹理，深层神经元可以看到完整物体。如果感受野不够大（网络太浅或核太小），深层神经元"看不到"足够大的区域，无法学习全局结构——这就是 GAN 的全局判别器需要多层下采样的原因。

### 2.3.7 Student t 分布

**定义：** 自由度为 $\nu$ 的 t 分布。$\nu=1$ 时退化为 Cauchy 分布：$f(x) = \frac{1}{\pi(1+x^2)}$。

**与高斯的对比：** $|x| > 3$ 时，高斯下降极快（密度接近 0），t 分布的密度仍然可观——这就是"重尾 (heavy tail)"。

**在 t-SNE 中的作用：** 低维用 t 分布（重尾）替代高斯（薄尾），给中距离点对足够的梯度信号，使其在低维中适度分开而非过度分散——这是 t-SNE 产生清晰簇分离的数学原因。